# binance-data-downloader (Python)

## 项目目标

> 从 [Binance Public Data](https://data.binance.vision/?prefix=data/futures/um/daily/klines/) 下载永续合约 K 线数据文件（保持原始交易所文件格式）
> 网站数据布局：`https://data.binance.vision`
>
> - `futures/`
>   - `cm/`（币本位交割合约，同下周结构）
>   - `um/`（U 本位永续）
>     - `daily/`
>       - `aggTrades/`
>       - `bookDepth/`
>       - `bookTicker/`
>       - `indexPriceKlines/`
>       - `klines/{symbol}/{timeframe}/{symbol}-{timeframe}-{yyyy-mm-dd}.zip`（本项目）
>       - `markPriceKlines/`
>       - `metrics/`
>       - `premiumIndexKlines/`
>       - `trades/`
>     - `monthly/`（同上按月打包）
> - `option/`（期权）
> - `spot/`（现货）
>
> 本地落盘：`{output_dir}/binance/data/um/daily/klines/{symbol}/{timeframe}/*.zip`
>
目标：按步骤实现数据下载全部功能（对标 Rust 版 `app/` 管线）。

## 配置

In [ ]:
# 配置（对标 config.rs，缺省与 Rust 版一致）
symbol = ["BTCUSDT"]  # 必填
timeframe = ["1d"]  # 必填
start = None  # 可选，缺省币种首日，形如 "2020-01-01"
end = None  # 可选，缺省昨天；未来值截到昨天
output_dir = "data"  # 可选
concurrency = 64  # 可选，必须≠0

print(symbol, timeframe, start, end, output_dir, concurrency)
# 效验（对标 validate）
VALID_TIMEFRAMES = ["1m", "3m", "5m", "15m", "30m", "1h", "2h", "4h", "6h", "8h", "12h", "1d"]
assert symbol, "symbol must not be empty"
assert timeframe, "timeframe must not be empty"
bad = [t for t in timeframe if t not in VALID_TIMEFRAMES]
assert not bad, f"unsupported timeframe: {bad}"
assert concurrency != 0, "concurrency must be greater than zero"
if start and end:
    assert start <= end, f"start {start} is after end {end}"
symbol, timeframe = sorted(set(symbol)), sorted(set(timeframe))  # 排序去重
print("validate ok:", symbol, timeframe)
# end 处理（对标 resolved_end：min(end,昨天)/缺省昨天）
from datetime import date, timedelta
yesterday = (date.today() - timedelta(days=1)).isoformat()
end = min(end, yesterday) if end else yesterday
print("end:", end)

## 获取开始时间

In [ ]:
# 开始时间（对标 get_first_kline_date + resolve_start，按币取一次 1m 首根）
import requests
from datetime import datetime, timezone

sym_starts = {}
for sym in symbol:
    url = f"https://fapi.binance.com/fapi/v1/klines?symbol={sym}&interval=1m&limit=1&startTime=0"
    k = requests.get(url, timeout=30).json()
    ms = k[0][0]
    sym_starts[sym] = datetime.fromtimestamp(ms / 1000, tz=timezone.utc).date().isoformat()

print(sym_starts)

## 构建下载链接

In [ ]:
# 组 URL（对标 01_build_urls：服务端前缀与本地前缀分离）
from datetime import date

SERVER_PREFIX = "data/futures/um/daily/klines"
LOCAL_PREFIX = "binance/data/um/daily/klines"

tasks = []  # (url, rel_path)
for sym in symbol:
    sym_start = sym_starts[sym]
    for tf in timeframe:
        cur = max(start, sym_start) if start else sym_start
        d0 = date.fromisoformat(cur)
        d1 = date.fromisoformat(end)
        while d0 <= d1:
            ds = d0.isoformat()
            fn = f"{sym}-{tf}-{ds}.zip"
            url = f"https://data.binance.vision/{SERVER_PREFIX}/{sym}/{tf}/{fn}"
            rel = f"{LOCAL_PREFIX}/{sym}/{tf}/{fn}"
            tasks.append((url, rel))
            d0 += timedelta(days=1)

print(len(tasks), tasks[0])

## 扫描本地去重

In [ ]:
# 扫描本地（对标 scan_local_files + build_download_tasks 过滤）
from pathlib import Path

local = set()
root = Path(output_dir)
if root.exists():
    for f in root.rglob("*"):
        if f.suffix == ".tmp":
            f.unlink(missing_ok=True)
        elif f.suffix == ".zip":
            local.add(str(f.relative_to(root)))

todo = [(u, r) for u, r in tasks if r not in local]
print(f"total={len(tasks)} local={len(local)} todo={len(todo)}")

## 执行下载

In [ ]:
# 下载（对标 03_download：有界并发 + 流式落盘 + 404 容忍 + 汇总）
from concurrent.futures import ThreadPoolExecutor
import time

ok = not_found = failed = 0

def fetch(task):
    url, rel = task
    dest = Path(output_dir) / rel
    for attempt in range(4):  # 指数退避重试
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 404:
                return "not_found"
            r.raise_for_status()
            dest.parent.mkdir(parents=True, exist_ok=True)
            tmp = dest.with_suffix(".zip.tmp")
            tmp.write_bytes(r.content)
            tmp.rename(dest)  # 原子落盘
            return "ok"
        except Exception:
            time.sleep(2 ** attempt)
    return "failed"

with ThreadPoolExecutor(max_workers=concurrency) as ex:
    for i, st in enumerate(ex.map(fetch, todo), 1):
        ok += st == "ok"
        not_found += st == "not_found"
        failed += st == "failed"
        if i % 100 == 0:
            print(f"{i}/{len(todo)} ok={ok} not_found={not_found} failed={failed}")

print(f"done: ok={ok} not_found={not_found} failed={failed}")

## 汇总

In [ ]:
# 汇总（对标 batch 汇总：失败即错 + 抽查落盘）
assert failed == 0, f"{failed}/{len(todo)} failed"
disk = [f for f in Path(output_dir).rglob("*.zip")]
print(f"tasks={len(tasks)} todo={len(todo)} ok={ok} not_found={not_found} disk_zips={len(disk)}")
print("sample:", sorted(str(f) for f in disk)[:2])